In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MapReduce_WordCount_TitanicNames") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")

25/12/26 13:58:09 WARN Utils: Your hostname, spark resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
25/12/26 13:58:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/26 13:58:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.csv("../data/titanic.csv", header=True, inferSchema=True)

# Sadece Name kolonunu al, null olanları çıkar
names_df = df.select("Name").dropna()
names_df.show(5, truncate=False)

+---------------------------------------------------+
|Name                                               |
+---------------------------------------------------+
|Braund, Mr. Owen Harris                            |
|Cumings, Mrs. John Bradley (Florence Briggs Thayer)|
|Heikkinen, Miss. Laina                             |
|Futrelle, Mrs. Jacques Heath (Lily May Peel)       |
|Allen, Mr. William Henry                           |
+---------------------------------------------------+
only showing top 5 rows



In [3]:
names_rdd = names_df.rdd.map(lambda row: row["Name"])
names_rdd.take(5)

['Braund, Mr. Owen Harris',
 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)',
 'Heikkinen, Miss. Laina',
 'Futrelle, Mrs. Jacques Heath (Lily May Peel)',
 'Allen, Mr. William Henry']

In [4]:
import re

words_rdd = names_rdd.flatMap(
    lambda name: re.sub(r"[^a-zA-Z\s]", " ", name.lower()).split()
)

words_rdd.take(20)

['braund',
 'mr',
 'owen',
 'harris',
 'cumings',
 'mrs',
 'john',
 'bradley',
 'florence',
 'briggs',
 'thayer',
 'heikkinen',
 'miss',
 'laina',
 'futrelle',
 'mrs',
 'jacques',
 'heath',
 'lily',
 'may']

In [5]:
word_counts = (words_rdd
               .map(lambda w: (w, 1))
               .reduceByKey(lambda a, b: a + b))

word_counts.take(10)

[('braund', 2),
 ('mr', 521),
 ('owen', 2),
 ('harris', 5),
 ('cumings', 1),
 ('mrs', 129),
 ('john', 44),
 ('bradley', 2),
 ('florence', 5),
 ('briggs', 1)]

In [6]:
top20 = word_counts.sortBy(lambda x: x[1], ascending=False).take(20)
top20

[('mr', 521),
 ('miss', 182),
 ('mrs', 129),
 ('william', 64),
 ('john', 44),
 ('master', 40),
 ('henry', 35),
 ('james', 24),
 ('charles', 24),
 ('george', 24),
 ('thomas', 22),
 ('mary', 20),
 ('edward', 18),
 ('anna', 17),
 ('joseph', 16),
 ('elizabeth', 15),
 ('johan', 15),
 ('frederick', 15),
 ('richard', 14),
 ('samuel', 13)]

In [7]:
top20_df = spark.createDataFrame(top20, ["word", "count"])
top20_df.show(20, truncate=False)

+---------+-----+
|word     |count|
+---------+-----+
|mr       |521  |
|miss     |182  |
|mrs      |129  |
|william  |64   |
|john     |44   |
|master   |40   |
|henry    |35   |
|james    |24   |
|charles  |24   |
|george   |24   |
|thomas   |22   |
|mary     |20   |
|edward   |18   |
|anna     |17   |
|joseph   |16   |
|elizabeth|15   |
|johan    |15   |
|frederick|15   |
|richard  |14   |
|samuel   |13   |
+---------+-----+

